# RoadScan v2 — YOLO detection experiment

This experiment starts from COCO-pretrained YOLO11s and uses exactly the 8,999 training images used by the ResNet-18 v2 baseline. Japan, India and Czech are training countries; China (1,977 images) is the validation domain. The United States (4,805 images) and local photographs are reserved for a separate final evaluation after the recipe and thresholds are fixed. No weights, predictions or processed labels from notebooks 04–13 are reused.

Detection mAP and image-level Macro-F1 answer different questions and are reported separately. Baseline reference: China Macro-F1 0.496, US Macro-F1 0.362. China results are development results, because checkpoint selection and threshold calibration use this set. No target score or absence of overfitting is guaranteed.

## 1. Validate and stage the fixed sample

The following cell checks image–label pairing, manifest equality, country counts, normalized YOLO boxes and exact image duplicates between training and validation. It copies data to local working storage for predictable training access. Labels remain unchanged. This audit does not exclude near-duplicate scenes; that remains a limitation.

In [ ]:
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib, json, shutil, time
import numpy as np
from PIL import Image
import yaml

BASE = Path('/kaggle/input/notebooks/sarvii7/roadscan-03-sample9000-v2')
SOURCE = BASE / 'sample9000'
ROOT = Path('/kaggle/working/sample9000')
ART = Path('/kaggle/working/artifacts_v2')
ART.mkdir(exist_ok=True)
CLASSES = ['D00', 'D10', 'D20', 'D40']
manifest = (BASE / 'sample9000_list.txt').read_text().splitlines()
assert len(manifest) == len(set(manifest)) == 8999
jobs = []
for split, expected in [('train', 8999), ('val', 1977)]:
    images = {p.stem: p for p in (SOURCE/split/'images').glob('*.jpg')}
    labels = {p.stem: p for p in (SOURCE/split/'labels').glob('*.txt')}
    assert images.keys() == labels.keys() and len(images) == expected
    if split == 'train':
        assert set(images) == set(manifest)
        assert Counter(s.split('_')[0] for s in images) == {'Japan':4000, 'India':3500, 'Czech':1499}
    for folder in ['images', 'labels']:
        (ROOT/split/folder).mkdir(parents=True, exist_ok=True)
    jobs += [(split, s, images[s], labels[s]) for s in sorted(images)]

def inspect_and_copy(job):
    split, stem, image, label = job
    raw = label.read_text()
    rows = [list(map(float, line.split())) for line in raw.splitlines() if line.strip()]
    for row in rows:
        assert len(row) == 5 and np.isfinite(row).all(), (split, stem, row)
        cls, x, y, w, h = row
        assert cls == int(cls) and 0 <= cls < 4
        assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
        assert x-w/2 >= -1e-5 and y-h/2 >= -1e-5 and x+w/2 <= 1.00001 and y+h/2 <= 1.00001
    data = image.read_bytes()
    target = ROOT/split/'images'/image.name
    target.write_bytes(data)
    with Image.open(target) as im:
        im.verify()
    (ROOT/split/'labels'/label.name).write_text(raw)
    return {'split':split, 'stem':stem, 'sha256':hashlib.sha256(data).hexdigest(), 'boxes':len(rows), 'classes':sorted({int(r[0]) for r in rows})}

started = time.monotonic()
records, errors = [], []
with ThreadPoolExecutor(max_workers=24) as pool:
    futures = {pool.submit(inspect_and_copy,j):j[:2] for j in jobs}
    for f in as_completed(futures):
        try:
            records.append(f.result())
        except Exception as e:
            errors.append({'image':futures[f], 'error':repr(e)})
        if (len(records)+len(errors)) % 500 == 0:
            print('Audited:', len(records)+len(errors), '/', len(jobs), flush=True)
(ART/'audit.json').write_text(json.dumps({'records':records, 'errors':errors}, indent=2))
assert not errors, 'Audit failed: see artifacts_v2/audit.json'
train_hash = {r['sha256'] for r in records if r['split']=='train'}
val_hash = {r['sha256'] for r in records if r['split']=='val'}
assert not train_hash & val_hash, 'Exact duplicate images cross train/validation. Resolve with team before training.'
shutil.copy2(BASE/'sample9000_list.txt', ART/'sample9000_list.txt')
DATA = Path('/kaggle/working/data_v2.yaml')
DATA.write_text(yaml.safe_dump({'path':str(ROOT),'train':'train/images','val':'val/images','names':dict(enumerate(CLASSES))}, sort_keys=False))
for split in ['train','val']:
    subset = [r for r in records if r['split']==split]
    print(split, 'images:',len(subset), 'boxes:',sum(r['boxes'] for r in subset), 'empty:',sum(r['boxes']==0 for r in subset))
print('Audit passed. Minutes:', round((time.monotonic()-started)/60,1), flush=True)


## 2. Fine-tune a COCO-pretrained detector

YOLO11s at 640 pixels provides a practical first run before testing larger resolutions. All trainable layers are fine-tuned using AdamW, cosine learning-rate decay, three warmup epochs and weight decay. Mosaic, horizontal flips, scale and HSV changes are applied to training images only. Small rotations are avoided because D00 and D10 depend on crack direction.

The run allows 40 epochs with early stopping after 8 epochs without validation improvement. A callback requests a stop at the next completed epoch once 90 minutes have elapsed; validation and artifact export add time. Epoch scheduling remains fixed. The checkpoint with the best validation detection fitness is selected. GPU availability is required. PyTorch is reused from Kaggle rather than reinstalled.

References: https://docs.ultralytics.com/models/yolo11/ and https://docs.ultralytics.com/modes/train/

In [ ]:
import sys, subprocess, importlib.metadata, platform
try:
    importlib.metadata.version('ultralytics')
except importlib.metadata.PackageNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.177'])
import torch, ultralytics
from ultralytics import YOLO
assert DATA.exists() and len(records) == 10976 and not errors
assert torch.cuda.is_available(), 'Select a GPU accelerator before running training.'
versions = {'python':platform.python_version(), 'torch':torch.__version__, 'ultralytics':ultralytics.__version__, 'gpu':torch.cuda.get_device_name(0)}
(ART/'environment.json').write_text(json.dumps(versions, indent=2))
print(versions)
model = YOLO('yolo11s.pt')
clock = {}
def start_clock(trainer):
    clock['start'] = time.monotonic()
def budget_stop(trainer):
    if time.monotonic() - clock['start'] >= 90*60:
        trainer.stop = True
        print('90-minute training budget reached; finishing this epoch and saving.')
model.add_callback('on_train_start', start_clock)
model.add_callback('on_fit_epoch_end', budget_stop)
train_results = model.train(
    data=str(DATA), imgsz=640, epochs=40, patience=8, batch=16,
    device=0, workers=2, seed=42, deterministic=True,
    optimizer='AdamW', lr0=0.001, lrf=0.01, cos_lr=True,
    warmup_epochs=3, weight_decay=0.0005,
    mosaic=1.0, close_mosaic=10, mixup=0.0,
    scale=0.5, translate=0.1, degrees=0.0, fliplr=0.5, flipud=0.0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    cache=False, amp=True, save=True, save_period=5, plots=True,
    project='/kaggle/working/runs_v2', name='yolo11s_640_seed42',
)
RUN = Path(model.trainer.save_dir)
BEST = RUN/'weights/best.pt'
assert BEST.exists()
shutil.copy2(BEST, ART/'yolo_v2_best.pt')
shutil.copy2(RUN/'results.csv', ART/'training_history.csv')
shutil.copy2(RUN/'args.yaml', ART/'training_args.yaml')
print('Best model saved:', ART/'yolo_v2_best.pt')


## 3. Evaluate detection and calibrate image-level decisions

Detection is evaluated on the complete China validation set using standard mAP50 and mAP50–95. For the ResNet comparison, each image receives four scores: the maximum retained detection confidence for each class, or zero when no detection is present. Predictions are retained down to confidence 0.001; this inference floor is fixed before evaluation.

Class thresholds are selected on China from 0.05 to 0.95 in steps of 0.05, matching the baseline grid. Both fixed-threshold (0.5) and calibrated Macro-F1 are saved, together with AUROC, per-class precision/recall/F1, and the always-positive reference. Calibrated validation F1 is optimistic and is not an independent test result. The saved thresholds must be reused unchanged on the United States and local photographs. Training history and country-specific training diagnostics will inform follow-up work; this run alone cannot establish generalization to every city.

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
best_model = YOLO(str(ART/'yolo_v2_best.pt'))
detection = best_model.val(data=str(DATA), split='val', imgsz=640, batch=16, device=0, workers=2, plots=True, project='/kaggle/working/evaluation_v2', name='china')
(ART/'china_detection.json').write_text(json.dumps({str(k):float(v) for k,v in detection.results_dict.items()}, indent=2))
stems = sorted(p.stem for p in (ROOT/'val/labels').glob('*.txt'))
truth = np.zeros((len(stems),4), dtype=int)
scores = np.zeros((len(stems),4), dtype=float)
lookup = {stem:i for i,stem in enumerate(stems)}
for i,stem in enumerate(stems):
    for line in (ROOT/'val/labels'/f'{stem}.txt').read_text().splitlines():
        if line.strip():
            truth[i,int(line.split()[0])] = 1
for result in best_model.predict(source=str(ROOT/'val/images'), imgsz=640, conf=0.001, iou=0.7, max_det=300, device=0, stream=True, verbose=False):
    i = lookup[Path(result.path).stem]
    for cls, conf in zip(result.boxes.cls.cpu().numpy().astype(int), result.boxes.conf.cpu().numpy()):
        scores[i,cls] = max(scores[i,cls], float(conf))
grid = np.arange(0.05,0.96,0.05)
thresholds = []
for c in range(4):
    values = [precision_recall_fscore_support(truth[:,c], scores[:,c]>=t, average='binary', zero_division=0)[2] for t in grid]
    thresholds.append(float(grid[int(np.argmax(values))]))
report = []
for mode, threshold in [('fixed_0.5', np.full(4,0.5)),('calibrated',np.array(thresholds))]:
    p,r,f,_ = precision_recall_fscore_support(truth,scores>=threshold,zero_division=0)
    for c,name in enumerate(CLASSES):
        prevalence = truth[:,c].mean()
        auc = roc_auc_score(truth[:,c],scores[:,c]) if len(np.unique(truth[:,c]))==2 else None
        report.append({'domain':'China','mode':mode,'class':name,'threshold':float(threshold[c]),'precision':float(p[c]),'recall':float(r[c]),'f1':float(f[c]),'auroc':auc,'always_positive_f1':float(2*prevalence/(1+prevalence))})
    print(mode,'Macro-F1:',round(float(f.mean()),4))
pd.DataFrame(report).to_csv(ART/'china_image_metrics.csv',index=False)
np.savez_compressed(ART/'china_image_scores.npz',stems=np.array(stems),truth=truth,scores=scores)
(ART/'inference_config.json').write_text(json.dumps({'classes':CLASSES,'thresholds':thresholds,'imgsz':640,'conf_floor':0.001,'iou':0.7,'max_det':300,'seed':42,'threshold_source':'China validation only'},indent=2))
print(pd.DataFrame(report).to_string(index=False))
print('Saved artifacts:',sorted(p.name for p in ART.iterdir()))
print('Next: freeze recipe and evaluate the US and local test images. No test scores were used here.')


## 4. Expanded-data experiment — CPU preparation (v3)

This independent section preserves the original v2 experiment above. Run only this section on CPU during preparation; do not use Run All (the earlier section trains v2).

Use all available Japan, India and Czech training images and a grouped 70/15/15 China MotorBike split for training, checkpoint validation and threshold calibration. US and local images remain outside development. The old baseline must be rerun on these exact manifests for a fair comparison.

Without verified route/scene metadata, filename blocks and perceptual hashes are conservative grouping heuristics, not proof of independent scenes. Export a visual review sheet and require review before training. No GPU training is started in this section.

### 4.1 Inspect the full source dataset


In [ ]:
from pathlib import Path
import json
V3_OUT = Path('/kaggle/working/roadscan_v3_split')
V3_OUT.mkdir(exist_ok=True)
roots = [p for p in Path('/kaggle/input').rglob('RoadScan_yolo') if p.is_dir()]
assert len(roots) == 1, f'Expected one full RoadScan_yolo source, found: {roots}'
V3_SOURCE = roots[0]
print('Full source:', V3_SOURCE)
for folder in sorted(V3_SOURCE.iterdir()):
    if folder.is_dir():
        print(folder.name, 'images:', len(list(folder.rglob('*.jpg'))), 'labels:', len(list(folder.rglob('*.txt'))))
print('Example image paths:')
for p in list(V3_SOURCE.rglob('*.jpg'))[:5]:
    print(p.relative_to(V3_SOURCE))


### 4.2 Audit images, group related candidates and export provisional manifests

CPU only. Audit the development images (including empty labels), preserve each 50-image China filename block, and merge perceptual near-duplicates (64-bit pHash Hamming distance <= 6) into a single group. Exact duplicates are grouped across all development countries. Group allocation targets 70/15/15 for China, using seed 42; actual ratios can differ. All non-China development images remain in training, so any connected China group is forced to training.

The code exports class counts, source-relative manifests for both models, YOLO path lists, candidate similar-image pairs and a contact sheet. These are provisional: perceptual similarity can miss related scenes or merge unrelated images. Verify against route metadata if available and review the sheet before training. Calibration is for thresholds only; it is not an independent final test.


In [ ]:
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import hashlib, re, random, csv, time
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from scipy.fft import dctn
import yaml

V3_CLASSES = ['D00', 'D10', 'D20', 'D40']
V3_COUNTRIES = ['Japan', 'India', 'Czech', 'China_MotorBike']
V3_SEED, V3_BLOCK, V3_RADIUS = 42, 50, 6
v3_start = time.monotonic()
def country_of(stem):
    return next((c for c in V3_COUNTRIES if stem.startswith(c + '_')), None)
def audit_image(p):
    country = country_of(p.stem)
    label = p.parent.parent/'labels'/(p.stem + '.txt')
    assert label.exists(), f'Missing label: {p}'
    rows = [list(map(float, s.split())) for s in label.read_text().splitlines() if s.strip()]
    present = np.zeros(4, dtype=int)
    for row in rows:
        assert len(row) == 5 and np.isfinite(row).all(), str(label)
        c,x,y,w,h = row
        assert c == int(c) and 0 <= c < 4 and 0 < w <= 1 and 0 < h <= 1
        assert x-w/2 >= -1e-4 and y-h/2 >= -1e-4 and x+w/2 <= 1.0001 and y+h/2 <= 1.0001, str(label)
        present[int(c)] = 1
    with Image.open(p) as im:
        im.load()
        a = np.asarray(im.convert('L').resize((32,32)), dtype=float)
        low = dctn(a, norm='ortho')[:8,:8].flatten()
        bits = low > np.median(low[1:])
        phash = int.from_bytes(np.packbits(bits).tobytes(), 'big')
    return dict(image=p, label=label, stem=p.stem, country=country, exact=hashlib.sha256(p.read_bytes()).hexdigest(), phash=phash, boxes=len(rows), present=present)
paths = sorted(p for p in V3_SOURCE.rglob('*.jpg') if country_of(p.stem))
assert paths and len({p.stem for p in paths}) == len(paths), 'Duplicate stems or no development images'
print('Auditing development images:', len(paths), flush=True)
with ThreadPoolExecutor(max_workers=4) as pool:
    v3_records = list(pool.map(audit_image, paths))
print('Audit passed:', dict(Counter(r['country'] for r in v3_records)), flush=True)
parent = list(range(len(v3_records)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def union(a,b):
    a,b = find(a),find(b)
    if a != b: parent[max(a,b)] = min(a,b)
exact_seen, blocks = {}, {}
for i,r in enumerate(v3_records):
    if r['exact'] in exact_seen: union(i, exact_seen[r['exact']])
    else: exact_seen[r['exact']] = i
    if r['country'] == 'China_MotorBike':
        match = re.search(r'(\d+)$', r['stem'])
        assert match, 'Need explicit grouping metadata for: ' + r['stem']
        block = int(match.group(1)) // V3_BLOCK
        if block in blocks: union(i,blocks[block])
        else: blocks[block] = i
# BK-tree searches all development hashes without an all-pairs distance matrix.
tree = None
near_pairs = []
for i,r in enumerate(v3_records):
    value = r['phash']
    if tree is None:
        tree = [value, i, {}]
        continue
    stack = [tree]
    while stack:
        node = stack.pop()
        distance = (value ^ node[0]).bit_count()
        if distance <= V3_RADIUS:
            union(i,node[1])
            near_pairs.append((i,node[1],distance))
        stack.extend(child for d,child in node[2].items() if distance-V3_RADIUS <= d <= distance+V3_RADIUS)
    node = tree
    while True:
        distance = (value ^ node[0]).bit_count()
        if distance == 0: break
        if distance not in node[2]:
            node[2][distance] = [value,i,{}]
            break
        node = node[2][distance]
groups = defaultdict(list)
for i in range(len(v3_records)): groups[find(i)].append(i)
forced = {g for g,ids in groups.items() if any(v3_records[i]['country'] != 'China_MotorBike' for i in ids)}
china_groups = sorted(set(groups)-forced)
random.Random(V3_SEED).shuffle(china_groups)
china_total = sum(r['country']=='China_MotorBike' for r in v3_records)
allocation = {g:'train' for g in forced}
counts = Counter()
targets = {'val':0.15*china_total,'calibration':0.15*china_total}
# Allocate whole groups; select the split with the greatest remaining relative deficit.
for g in china_groups:
    split = max(targets, key=lambda s:(targets[s]-counts[s])/targets[s])
    if counts[split] < targets[split]:
        allocation[g] = split
        counts[split] += len(groups[g])
    else: allocation[g] = 'train'
manifest_rows = []
for i,r in enumerate(v3_records):
    split = allocation[find(i)]
    row = dict(split=split, country=r['country'], stem=r['stem'], group=find(i), image=str(r['image'].relative_to(V3_SOURCE)), label=str(r['label'].relative_to(V3_SOURCE)), sha256=r['exact'], phash=f"{r['phash']:016x}", boxes=r['boxes'])
    row.update({c:int(r['present'][j]) for j,c in enumerate(V3_CLASSES)})
    manifest_rows.append(row)
v3_df = pd.DataFrame(manifest_rows)
assert all(len(set(v3_df.loc[v3_df['group']==g,'split']))==1 for g in groups)
assert all(allocation[find(a)]==allocation[find(b)] for a,b,d in near_pairs)
assert set(v3_df['split']) == {'train','val','calibration'}, 'Grouping collapsed a split; review metadata before proceeding'
assert (v3_df.groupby('split')[V3_CLASSES].sum() > 0).all().all(), 'A class is absent; redesign grouped split'
v3_df.to_csv(V3_OUT/'manifest.csv',index=False)
for split in ['train','val','calibration']:
    part = v3_df[v3_df.split==split]
    part.to_csv(V3_OUT/(split+'_manifest.csv'),index=False)
    (V3_OUT/(split+'.txt')).write_text('\n'.join(str(V3_SOURCE/p) for p in part.image)+'\n')
config = dict(path=str(V3_SOURCE),train=str(V3_OUT/'train.txt'),val=str(V3_OUT/'val.txt'),names=dict(enumerate(V3_CLASSES)))
(V3_OUT/'data_v3.yaml').write_text(yaml.safe_dump(config,sort_keys=False))
summary = v3_df.groupby(['split','country']).agg(images=('stem','size'),boxes=('boxes','sum'),**{c:(c,'sum') for c in V3_CLASSES})
summary.to_csv(V3_OUT/'class_distribution.csv')
pair_rows = [dict(image_a=v3_records[a]['stem'],image_b=v3_records[b]['stem'],hamming=d,group=find(a),split=allocation[find(a)]) for a,b,d in near_pairs]
pd.DataFrame(pair_rows,columns=['image_a','image_b','hamming','group','split']).to_csv(V3_OUT/'near_duplicate_pairs.csv',index=False)
# Reproducible examples nearest the pHash boundary, for visual inspection.
examples = sorted(near_pairs,key=lambda t:(-t[2],v3_records[t[0]]['stem']))[:12]
sheet = Image.new('RGB',(640,max(1,len(examples))*160),'white')
draw = ImageDraw.Draw(sheet)
for k,(a,b,d) in enumerate(examples):
    for col,idx in enumerate([a,b]):
        with Image.open(v3_records[idx]['image']) as im:
            tile = ImageOps.contain(im.convert('RGB'),(310,130))
        sheet.paste(tile,(col*320,k*160))
        draw.text((col*320,k*160+132),v3_records[idx]['stem'][:40],fill='black')
    draw.text((0,k*160+146),f'Hamming={d}; assigned={allocation[find(a)]}',fill='black')
sheet.save(V3_OUT/'similarity_review.jpg')
report = dict(seed=V3_SEED,source=str(V3_SOURCE),images=len(v3_df),groups=len(groups),china_images=china_total,china_filename_block_size=V3_BLOCK,phash_radius=V3_RADIUS,near_pairs=len(near_pairs),cross_split_exact_or_detected_near_pairs=0,split_counts=v3_df.groupby(['split','country']).size().to_dict().__str__(),status='PROVISIONAL_REQUIRES_SCENE_REVIEW',test_data_accessed=False,seconds=round(time.monotonic()-v3_start,1))
(V3_OUT/'audit.json').write_text(json.dumps(report,indent=2))
(V3_OUT/'README.txt').write_text('Provisional grouped split. Review similarity_review.jpg and route/scene metadata before training. No proof of scene independence. Re-run ResNet on exactly these manifests. Use val for checkpoint selection and calibration only for thresholds. US/local remain held out. Source-relative manifests are portable; regenerate absolute YOLO lists when the Kaggle mount changes. Previous China evaluation means these are development subsets, not fresh tests. No training executed.')
print(summary.to_string())
print(json.dumps(report,indent=2))
print('Saved:',sorted(p.name for p in V3_OUT.iterdir()))
